<a href="https://colab.research.google.com/github/SkyFlowTraveler/FC-MLP_on_NumPy/blob/main/Iris_selfmade_MLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from sklearn.datasets import load_iris
import numpy as np

iris = load_iris()

X = iris.data
y = iris.target

In [ ]:
X = np.c_[X, y]

In [ ]:
X.shape

(150, 5)

In [ ]:
np.unique(y, return_counts = True)

(array([0, 1, 2]), array([50, 50, 50]))

In [ ]:
#Архитектура будет 4 ReLU -> 4 ReLU -> 3 softmax
#Следовательно, при использовании ReLU мне нужно использовать инициализацию Kaiming He, а на последнем просто из возможности использую Xavier, хотя можно и He
#Главное, перед softmax сделать по вектору выходов: z = z - max(z), чтобы exp() не улетела.

In [ ]:
np.set_printoptions(suppress=True, precision=15)

def relu(x):
  return np.maximum(0, x)
def softmax(vector):
  print(vector.shape, np.max(vector, axis = 1).shape)
  vector = vector - np.max(vector, axis = 1, keepdims = True)
  exp_vector = np.exp(vector)
  return exp_vector/np.sum(exp_vector, axis = 1, keepdims = True)


#проверка функции softmax

#vec = np.array([1, 2, 3])

#print(softmax(vec))

In [ ]:
def Kaiming_He(inp, out, neurons): #Начнём с He, нам нужна дисперсия 2/N_inp, следовательно стандартное отклонение sqrt(2)/sqrt(N_inp)
  W = np.random.randn(neurons, inp)*(np.sqrt(2/inp))
  return W
def Xavier(inp, out, neurons):
  W = np.random.randn(neurons, inp)*np.sqrt(2/(inp+out))
  return W

In [ ]:
class layer():
  def __init__(self, inp, neurons, out, activ):
    #сохраним значения для backward-pass
    if activ == relu:
      self.weight_vec = Kaiming_He(inp, out, neurons)
    else:
      self.weight_vec = Xavier(inp, out, neurons)  #размерность lay.weight_vec - (количество нейронов в слое x количество входов в слой)
    self.biases = np.zeros(neurons)
    self.inp_vec = np.zeros(inp*150).reshape(150, inp)
    self.sum_wz_b = np.zeros(neurons* 150).reshape(150, neurons) #w - веса, z - входы, сумма wz это значение перед использованием функции активации.
    self.output = np.zeros(neurons* 150).reshape(150, neurons)
    self.activ = activ

class loss_layer():
  def __init__(self, func):
    self.func = func

  def cr_ent(self, prev_layer_output_vector, y_true):
    cr_arr = np.zeros(150).reshape(150, 1)
    for i in range(len(prev_layer_output_vector)):
      #print(cr_arr, prev_layer_output_vector, i, y_true)
      cr_arr[i] = -np.log(prev_layer_output_vector[i, int(y_true[i])])
    print(np.sum(cr_arr),np.mean(cr_arr))
    return cr_arr

l_1 = layer(4, 4, 4, relu)
l_2 = layer(4, 4, 4, relu)
l_3 = layer(4, 3, 1, softmax)
loss_l = loss_layer('cr_ent')

list_of_layers = (l_1, l_2, l_3, loss_l)

In [ ]:
def forward_pass(X, list_of_layers):
  soft_acted = np.empty((0,3))
  loss = 0
  y_true = X[:,-1]
  X = X[:, :-1]
  for lay in list_of_layers:
      #print('layer:', lay)
      if lay is not loss_l:
        lyr_fwd_pass(X, lay)
        X = lay.output
        #print('OUTPUT:', lay.output)
      else:
        #print(prev_lay.output, y_true)
        soft_acted = np.vstack([soft_acted, prev_lay.output])
        loss += lay.cr_ent(prev_lay.output, y_true)
      prev_lay = lay
  print('loss shape:', loss.shape)
  #if flag == True:
    #print(lay.output[:out_size])
  return soft_acted

def lyr_fwd_pass(input, layer):
  #print(input.shape, layer.weight_vec.shape, layer.biases) #должно быть где-то (150, 4)
  layer.sum_wz_b = (input@layer.weight_vec.T) + layer.biases
  layer.input = input
  if layer.activ == relu:
    #print('ПЕРЕД АКТИВАЦИЕЙ:', layer.sum_wz_b)
    layer.output = relu(layer.sum_wz_b)
  elif layer.activ == softmax:
    layer.output = softmax(layer.sum_wz_b)
  #print(layer.output.shape)
  return layer.output

def backward_pass(soft_acted, lr): #выберем AdaptiveMoments(Adam). P.S. Когда-нибудь сделаю апгрейд с computational graph, чтобы был универсальный алгоритм
  y_true = np.zeros(450).reshape(150, 3)
  for index, row in enumerate(X):
    y_true[index, int(row[-1])]+=1
  z_softmax = (soft_acted - y_true)/150 #after cross-entropy + softmax

  dl_dz = z_softmax

  for i in range(len(list_of_layers)-2, -1, -1): #с 3 линейного слоя и до 1 в данном случае.
    lay = list_of_layers[i]
    if i>0:
      next_lay = list_of_layers[i-1]
    #print(lay.biases.shape, dl_dz.shape)
    lay.biases -= lr * dl_dz.sum(axis=0)                                  #(lay.sum_wz_b > 0)
    prev_dz = dl_dz
    #print(dl_dz.shape, lay.weight_vec.shape, next_lay.sum_wz_b.shape)
    if i>=1:
      dl_dz = dl_dz @ lay.weight_vec * (next_lay.sum_wz_b > 0)                                   #dl_dz * (lay.sum_wz_b > 0) * lay.weight_vec
    #print(prev_dz.shape, lay.input.T.shape)
    lay.weight_vec -= lr * (prev_dz.T @ lay.input)

'''
def adam(mbatch, epochs):
  for lay in list_of_layers:
    for row in X:
      loss = forward_pass(X, lay)

  for i in range(2, -1, -1):
    backward_pass(loss, list_of_layers[i])
'''


'\ndef adam(mbatch, epochs):\n  for lay in list_of_layers:\n    for row in X:\n      loss = forward_pass(X, lay)\n\n  for i in range(2, -1, -1):\n    backward_pass(loss, list_of_layers[i])\n'

In [ ]:
lr = 0.1
n = 150
sf_act = forward_pass(X, list_of_layers)
for i in range(n):
  sf_act = forward_pass(X, list_of_layers)
  backward_pass(sf_act, lr)
sf_act = forward_pass(X, list_of_layers)
print(sf_act[:150])

(150, 3) (150,)
1732.043986846267 11.546959912308447
loss shape: (150, 1)
(150, 3) (150,)
1732.043986846267 11.546959912308447
loss shape: (150, 1)
(150, 3) (150,)
185.37158697731192 1.2358105798487462
loss shape: (150, 1)
(150, 3) (150,)
171.7523321101899 1.145015547401266
loss shape: (150, 1)
(150, 3) (150,)
167.1367753037993 1.1142451686919954
loss shape: (150, 1)
(150, 3) (150,)
165.5294325012257 1.1035295500081714
loss shape: (150, 1)
(150, 3) (150,)
162.98524258958486 1.0865682839305657
loss shape: (150, 1)
(150, 3) (150,)
158.5438141569664 1.0569587610464426
loss shape: (150, 1)
(150, 3) (150,)
151.67596187273335 1.0111730791515556
loss shape: (150, 1)
(150, 3) (150,)
145.8702359241578 0.9724682394943854
loss shape: (150, 1)
(150, 3) (150,)
142.92189884204544 0.9528126589469695
loss shape: (150, 1)
(150, 3) (150,)
141.38391029604293 0.9425594019736195
loss shape: (150, 1)
(150, 3) (150,)
139.68751184537626 0.9312500789691751
loss shape: (150, 1)
(150, 3) (150,)
137.7084056850447